# 第9周｜病历多表与SQL

学习目标：主键、外键、一对多、患者数与记录数。

__数据：默认使用官方 PRIME-CVD 5 万人数据__（`python run.py download` 下载，来源与 MD5 核验记录在 `data/raw/manifest.json`）。把 `MODE` 改成 `"demo"` 可以切换到离线演示数据，但演示数据是另外生成的，**不是 PRIME-CVD**，结果不能当作官方结果。

本周验收：说明原始多表直接连接为何可能产生笛卡尔扩张。

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

# 从项目根目录、notebooks目录或其他项目内目录启动均可。
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run.py").exists()), None)
if ROOT is None:
    raise RuntimeError("请在解压后的项目目录中启动 Jupyter。")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from primecvd.common import load_config, provenance_label
from primecvd.pipeline import prepare
from primecvd.splitting import select_split

MODE = "official"  # 官方数据；离线练习或跑测试时可改为 "demo"（演示数据不是 PRIME-CVD）
cfg = load_config(ROOT / "configs" / f"{MODE}.json")
cfg["output_dir"] = f"outputs/notebooks_{MODE}"
print(provenance_label(cfg))
df, splits = prepare(cfg)
train = select_split(df, splits, "train")
valid = select_split(df, splits, "valid")
# 不创建test DataFrame；模型试验只使用train和valid。
print("队列、训练集、验证集：", len(df), len(train), len(valid))

# 命令行绘图模块采用无窗口后端；在Notebook中显式启用行内图像。
get_ipython().run_line_magic("matplotlib", "inline")
from primecvd import pubplot as pp   # 与精讲篇相同的顶刊作图风格（在启用行内图像之后设置）
pp.setup()

OFFICIAL SOURCE FILES / 官方下载文件
队列、训练集、验证集： 50000 35000 7500


## 为什么多张病历表不能直接连起来？
一个人可以有多条慢性病记录和多条检验记录。直接连接两个一对多表可能使记录相乘。先聚合到患者层，再与主表连接，是本周的基本练习。

In [2]:
from primecvd.data import load_emr
tables = load_emr(cfg)
display(pd.DataFrame([{"table":name, "rows":len(t), "unique_patients":t.Patient_ID.nunique()}
                      for name,t in tables.items()]))
for name, t in tables.items():
    print(name)
    display(t.head(3))

,table,rows,unique_patients
0,master,50000,50000
1,diseases,4417,4271
2,measurements,210000,50000


master


,Patient_ID,Age_At_2024,SMOKING_STATUS,IRSD_Quintile,CVD_Event,CVD_Time
0,269,57.40,non,4,1,2019-12
1,272,46.23,ex,3,0,2022-12
2,281,62.49,non,5,0,2022-12


diseases


,Patient_ID,Category,Date
0,1338163469,Diabetes,2012-05
1,150591944,Diabetes,2014-12
2,545940569,ICD10: E11,2016-10


measurements


,Patient_ID,Value,Description,Date,Unit
0,819127997,78.08730821809296,eGFR,2014-03,mL/min/1.73m²
1,3847713176,8.369465931621287,HbA1c,2015-02,%
2,3508304696,140.32800536506784,SBP,2016-09,mmHg


In [3]:
import sqlite3
with sqlite3.connect(":memory:") as connection:
    for name,t in tables.items():
        t.to_sql(name, connection, index=False, if_exists="replace")
    query = (ROOT/"sql/01_patient_level_summary.sql").read_text(encoding="utf-8")
    patient_summary = pd.read_sql_query(query, connection)
display(patient_summary.head())
assert len(patient_summary) == len(tables["master"])
assert patient_summary.Patient_ID.is_unique
print("SQL连接后患者数不变。")

,Patient_ID,Age_At_2024,n_disease_records,n_measurement_records
0,269,57.40,0,5
1,272,46.23,0,5
2,281,62.49,0,4
3,296,58.91,0,4
4,317,54.09,0,4


SQL连接后患者数不变。


## 独立练习
选择一名同时有多条诊断和检验记录的人，手算直接连接会生成多少行。说明“总行数未变”和“唯一患者数未变”为什么是不同检查。病历日期是作者构造的记录形式，不能直接解释为真实发病顺序。

---
本周复盘：请用自己的话记录一个学会的概念、一个发现的错误和一个待解决问题。

特别提醒：每次合并前后核对患者数；不按表行号盲目拼接。